## 1. Конвертація predictions → binary direction


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_curve

from src.pipelines.evaluation_pipeline import evaluate_experiment_b, load_all_predictions

processed_dir = Path('data/processed')
preds = load_all_predictions(processed_dir)
test_df = pd.read_parquet(processed_dir / 'test_returns.parquet')
y_test = test_df['brent_return']

binary_preds = {k: (v > 0).astype(int) for k, v in preds.items()}
y_binary = (y_test > 0).astype(int)

## 2. Platt scaling калібрація для моделей без нативних probability


In [ ]:
platt_probs = {}
for name, series in preds.items():
    scores = np.abs(series.to_numpy(dtype=float)).reshape(-1, 1)
    y = y_binary.loc[series.index].to_numpy()
    model = LogisticRegression(max_iter=1000)
    model.fit(scores, y)
    platt_probs[name] = model.predict_proba(scores)[:, 1]

## 3. Таблиця метрик Експерименту B (MCC, F1, ROC-AUC per model)


In [ ]:
results_b = evaluate_experiment_b(preds, y_test)
results_b

## 4. Confusion matrices для топ-3 моделей (subplot grid)


In [ ]:
top_models = results_b.sort_values('MCC', ascending=False).head(3).index.tolist()
fig = make_subplots(rows=1, cols=3, subplot_titles=top_models)
for idx, model in enumerate(top_models):
    y_true = y_binary.loc[preds[model].index]
    y_pred = binary_preds[model]
    cm = confusion_matrix(y_true, y_pred)
    fig.add_trace(go.Heatmap(z=cm, showscale=False), row=1, col=idx + 1)
fig.update_layout(title='Confusion Matrices')
fig.show()

## 5. ROC curves overlay (всі моделі на одному графіку)


In [ ]:
fig = go.Figure()
for name, probs in platt_probs.items():
    y_true = y_binary.loc[preds[name].index]
    fpr, tpr, _ = roc_curve(y_true, probs)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=name))
fig.update_layout(title='ROC Curves', xaxis_title='FPR', yaxis_title='TPR')
fig.show()

## 6. Аналіз: чи є кореляція між MASE (Exp A) та MCC (Exp B)?


In [ ]:
results_a = pd.read_csv(processed_dir / 'results_experiment_a_h1.csv', index_col=0)
merged = results_a.join(results_b, how='inner')
fig = px.scatter(merged, x='MASE', y='MCC', text=merged.index, title='MASE vs MCC')
fig.show()